# Tests for linear Hilbert-space solvers

This notebook tests four combinations of real/complex and self-adjoint/non-self-adjoint operators. Implementation-equivalence checks use three iterations. For approximation tests, Landweber uses 300 iterations, while Richardson, the $\nu$-methods, and CGNE use at most 50. The inner CG iteration of Tikhonov regularization is controlled by its own stopping rules and may use up to 1000 iterations.

Strict implementation-equivalence tests use a relative tolerance of $10^{-12}$. Approximation tests require a relative domain error of at most 1% and a relative codomain error of at most 0.1%. Iteration limits are passed through the solver's `max_it` convenience argument.

In [1]:
import logging
import numpy as np

from regpy.solvers.setting import Setting
from regpy.solvers.linear.cgne import CGNE
from regpy.solvers.linear import (
    Landweber, Richardson, NuMethod, NuMethodSelfAdjoint,
)
from regpy.solvers.linear.tikhonov import TikhonovCG
from regpy.solvers.linear.spectral import (
    Spectral_Landweber, SpectralSelfAdjoint_Richardson,
    Spectral_Tikhonov,
)
from regpy.operators import (
    ExponentialConvolution, PeriodicShift, PtwMultiplication,
)
from regpy.vecsps import UniformGridFcts
from regpy.hilbert import L2
from regpy.stoprules import CountIterations

logging.getLogger().setLevel(logging.WARNING)
logging.disable(logging.INFO)

LANDWEBER_STEPS = 300
OTHER_ITERATION_STEPS = 200
EQUIVALENCE_STEPS = 3
TIKHONOV_MAX_IT = 1000
STRICT_TOL = 1e-12
DOMAIN_ERROR_LIMIT = 1e-2
CODOMAIN_ERROR_LIMIT = 1e-3
REGPAR = 1e-4
NU = 1.0
STEPSIZE_SAFETY = 1.0
TIKHONOV_EQ_RELTOL = 1e-12
TIKHONOV_ABS_TOL = 1e-4
TIKHONOV_REL_TOL = 1e-4
ALL_FAILURES = []

In [2]:
grid = UniformGridFcts((-1, 1, 100), periodic=True)
complex_grid = grid.complex_space()
pts = grid.coords[0]

xdag = 1.0 / (1 - 0.4*np.cos(5*np.pi*pts) + 0.3*np.cos(np.pi*pts)) + 1.0
xdag_co = (
    1.0 / (1 - 0.4*np.exp(5*np.pi*1j*pts) + 0.3*np.cos(np.pi*pts))
    + 1.0
)

sa_real_op = 50.*ExponentialConvolution(grid, a=0.05)
real_op = (
    50*ExponentialConvolution(grid, a=0.05)
    * PeriodicShift(grid, shift=(0.3,), self_adjoint=False)
)
sa_complex_op = 0.02*ExponentialConvolution(complex_grid, a=0.05)
complex_op = (
    50*ExponentialConvolution(complex_grid, a=0.02)
    * PtwMultiplication(complex_grid, 1./(1.-0.2*np.exp(1j*np.pi*pts)))
)

CASES = [
    {"name": "real, self-adjoint", "op": sa_real_op, "x_exact": xdag, "self_adjoint": True},
    {"name": "real, non-self-adjoint", "op": real_op, "x_exact": xdag, "self_adjoint": False},
    {"name": "complex, self-adjoint", "op": sa_complex_op, "x_exact": xdag_co, "self_adjoint": True},
    {"name": "complex, non-self-adjoint", "op": complex_op, "x_exact": xdag_co, "self_adjoint": False},
]

for case in CASES:
    case["data"] = case["op"](case["x_exact"])
    case["init"] = case["op"].domain.zeros()
    assert case["op"].self_adjoint == case["self_adjoint"], (
        f'{case["name"]}: unexpected self_adjoint flag'
    )

[(case["name"], case["op"].self_adjoint) for case in CASES]

[('real, self-adjoint', True),
 ('real, non-self-adjoint', False),
 ('complex, self-adjoint', True),
 ('complex, non-self-adjoint', False)]

In [3]:
def make_setting(case, *, codomain_free=False, regpar=None):
    return Setting(
        op=case["op"], penalty=L2, data_fid=L2,
        init=case["init"], incorporate_init_in_penalty=False,
        data=case["data"], regpar=regpar,
        codomain_free=codomain_free, explicitDataWarning=False,
    )


def difference_norm(hspace, left, right):
    difference = hspace.vecsp.empty()
    hspace.vecsp.copyto(difference, left)
    difference -= right
    return hspace.norm(difference)


def relative_difference(hspace, left, right):
    denominator = max(
        hspace.norm(left), hspace.norm(right), np.finfo(float).tiny,
    )
    return difference_norm(hspace, left, right) / denominator


def run_steps(solver, max_steps=OTHER_ITERATION_STEPS):
    approximation, _ = solver.run(max_it=max_steps)
    return approximation, solver.iteration_step_nr


def applicability(solver_class, setting):
    result, _ = solver_class.check_applicability(setting)
    return result


def require_applicable(solver_class, setting, label):
    result = applicability(solver_class, setting)
    if not result["applicable"]:
        raise AssertionError(f'{label} unexpectedly inapplicable: {result["info"]}')
    return result


def record_strict_comparison(failures, label, hspace, left, right):
    error = relative_difference(hspace, left, right)
    print(f'{label:<72} {error:.3e}')
    if not error < STRICT_TOL:
        failures.append(f'{label}: {error:.3e} >= {STRICT_TOL:.1e}')
    return error


for case in CASES:
    standard_setting = make_setting(case)
    case["h_domain"] = standard_setting.h_domain
    case["h_codomain"] = standard_setting.h_codomain
    # case["op"].set_gram_factor(standard_setting.h_codomain)
    # Forward norm ||T|| for Landweber and the standard nu-method.
    case["op_norm"] = case["op"].norm(
        standard_setting.h_domain, standard_setting.h_codomain
    )
    if case["self_adjoint"]:
        # Norm of G_X^{-1} A for Richardson-type methods, where A
        # is interpreted as a self-adjoint map X -> X*.
        case["self_adjoint_op_norm"] = case["op"].norm_sa(
            standard_setting.h_domain
        )
    case["landweber_stepsize"] = STEPSIZE_SAFETY / case["op_norm"]**2

print('Estimated operator norms:')
for case in CASES:
    print(f'  {case["name"]:<30} {case["op_norm"]:.8g}')

Estimated operator norms:
  real, self-adjoint             50
  real, non-self-adjoint         50
  complex, self-adjoint          0.02
  complex, non-self-adjoint      61.021839


## Codomain-free versus standard formulations

Each standard/codomain-free pair is run for three steps with a shared step size. Both domain iterates and their evaluated images are compared. The longer method-specific iteration counts are used only for the approximation-quality tests below.

In [4]:
strict_failures = []
ITERATIVE_SOLUTIONS = {}

for case in CASES:
    print(f'\n{case["name"]}')
    standard = make_setting(case)
    codomain_free = make_setting(case, codomain_free=True)
    mu = case["landweber_stepsize"]

    require_applicable(Landweber, standard, 'Landweber')
    require_applicable(Landweber, codomain_free, 'codomain-free Landweber')
    lw_standard = Landweber(
        standard, stepsize=mu, op_norm=case["op_norm"],
        change_init=False, logging_level=logging.WARNING,
    )
    lw_free = Landweber(
        codomain_free, stepsize=mu, op_norm=case["op_norm"],
        change_init=False, logging_level=logging.WARNING,
    )
    x_lw_standard, lw_steps_standard = run_steps(
        lw_standard, max_steps=EQUIVALENCE_STEPS
    )
    x_lw_free, lw_steps_free = run_steps(
        lw_free, max_steps=EQUIVALENCE_STEPS
    )
    ITERATIVE_SOLUTIONS[case["name"]] = {
        "Landweber standard": x_lw_standard,
        "Landweber codomain-free": x_lw_free,
    }
    record_strict_comparison(
        strict_failures, '  Landweber: domain', case["h_domain"],
        x_lw_standard, x_lw_free,
    )
    record_strict_comparison(
        strict_failures, '  Landweber: codomain', case["h_codomain"],
        case["op"](x_lw_standard), case["op"](x_lw_free),
    )

    require_applicable(NuMethod, standard, 'nu-method')
    require_applicable(NuMethod, codomain_free, 'codomain-free nu-method')
    nu_standard = NuMethod(
        standard, nu=NU, stepsize=mu, op_norm=case["op_norm"],
        change_init=False, logging_level=logging.WARNING,
    )
    nu_free = NuMethod(
        codomain_free, nu=NU, stepsize=mu, op_norm=case["op_norm"],
        change_init=False, logging_level=logging.WARNING,
    )
    x_nu_standard, nu_steps_standard = run_steps(
        nu_standard, max_steps=EQUIVALENCE_STEPS
    )
    x_nu_free, nu_steps_free = run_steps(
        nu_free, max_steps=EQUIVALENCE_STEPS
    )
    ITERATIVE_SOLUTIONS[case["name"]].update({
        "nu-method standard": x_nu_standard,
        "nu-method codomain-free": x_nu_free,
    })
  
    record_strict_comparison(
        strict_failures, '  nu-method: domain', case["h_domain"],
        x_nu_standard, x_nu_free,
    )
    record_strict_comparison(
        strict_failures, '  nu-method: codomain', case["h_codomain"],
        case["op"](x_nu_standard), case["op"](x_nu_free),
    )

    tikh_standard_setting = make_setting(case, regpar=REGPAR)
    tikh_free_setting = make_setting(case, codomain_free=True, regpar=REGPAR)
    require_applicable(TikhonovCG, tikh_standard_setting, 'TikhonovCG')
    require_applicable(TikhonovCG, tikh_free_setting, 'codomain-free TikhonovCG')
    tikh_standard = TikhonovCG(
        tikh_standard_setting, tol=None, reltolx=TIKHONOV_EQ_RELTOL,
        reltoly=None, logging_level=logging.WARNING,
    )
    tikh_free = TikhonovCG(
        tikh_free_setting, tol=None, reltolx=TIKHONOV_EQ_RELTOL,
        reltoly=None, logging_level=logging.WARNING,
    )
    x_tikh_standard, _ = tikh_standard.run(max_it=EQUIVALENCE_STEPS)
    x_tikh_free, _ = tikh_free.run(max_it=EQUIVALENCE_STEPS)
    print(
        f'  Tikhonov-CG iterations: standard={tikh_standard.iteration_step_nr}, '
        f'codomain-free={tikh_free.iteration_step_nr}'
    )
    record_strict_comparison(
        strict_failures, '  Tikhonov-CG: domain', case["h_domain"],
        x_tikh_standard, x_tikh_free,
    )
    record_strict_comparison(
        strict_failures, '  Tikhonov-CG: codomain', case["h_codomain"],
        case["op"](x_tikh_standard), case["op"](x_tikh_free),
    )

if strict_failures:
    ALL_FAILURES.extend(
        'strict equivalence: ' + failure for failure in strict_failures
    )
    print('\nStrict equivalence failures were recorded; continuing.')
else:
    print('\nAll standard/codomain-free comparisons passed.')


real, self-adjoint
  Landweber: domain                                                      4.574e-16
  Landweber: codomain                                                    1.581e-16
  nu-method: domain                                                      8.302e-16
  nu-method: codomain                                                    3.406e-16
  Tikhonov-CG iterations: standard=3, codomain-free=3
  Tikhonov-CG: domain                                                    1.188e-15
  Tikhonov-CG: codomain                                                  2.393e-16

real, non-self-adjoint
  Landweber: domain                                                      6.964e-16
  Landweber: codomain                                                    2.693e-16
  nu-method: domain                                                      1.102e-15
  nu-method: codomain                                                    5.223e-16
  Tikhonov-CG iterations: standard=3, codomain-free=3
  Tikhonov-CG: dom

## Landweber and Richardson versus their spectral filters

Each spectral filter uses the same step size $\mu$ as its iterative counterpart. These implementation-equivalence checks use three steps. Standard and codomain-free formulations are compared separately, and Richardson is tested only for the self-adjoint operators.

In [5]:
spectral_iteration_failures = []

for case in CASES:
    print(f'\n{case["name"]}')
    mu = case["landweber_stepsize"]
    standard = make_setting(case)
    codomain_free = make_setting(case, codomain_free=True)
    require_applicable(Spectral_Landweber, standard, 'spectral Landweber')
    require_applicable(
        Spectral_Landweber, codomain_free, 'codomain-free spectral Landweber'
    )
    spectral_standard = Spectral_Landweber(
        standard, k=EQUIVALENCE_STEPS, stepsize=mu
    )
    spectral_free = Spectral_Landweber(
        codomain_free, k=EQUIVALENCE_STEPS, stepsize=mu
    )
    x_spectral_standard, _ = run_steps(spectral_standard, 1)
    x_spectral_free, _ = run_steps(spectral_free, 1)
    x_lw_standard = ITERATIVE_SOLUTIONS[case["name"]]["Landweber standard"]
    x_lw_free = ITERATIVE_SOLUTIONS[case["name"]]["Landweber codomain-free"]
    record_strict_comparison(
        spectral_iteration_failures,
        '  Landweber standard: iterative vs spectral',
        case["h_domain"], x_lw_standard, x_spectral_standard,
    )
    record_strict_comparison(
        spectral_iteration_failures,
        '  Landweber codomain-free: iterative vs spectral',
        case["h_domain"], x_lw_free, x_spectral_free,
    )

    if not case["self_adjoint"]:
        print('  Richardson: SKIP (operator is not self-adjoint)')
        continue

    richardson_stepsize = (
        STEPSIZE_SAFETY / case["self_adjoint_op_norm"]
    )
    require_applicable(Richardson, standard, 'Richardson')
    require_applicable(
        Richardson, codomain_free, 'codomain-free Richardson'
    )
    require_applicable(
        SpectralSelfAdjoint_Richardson, standard,
        'spectral Richardson',
    )
    require_applicable(
        SpectralSelfAdjoint_Richardson, codomain_free,
        'codomain-free spectral Richardson',
    )

    richardson_standard = Richardson(
        standard, stepsize=richardson_stepsize,
        op_norm=case["self_adjoint_op_norm"],
        change_init=False, logging_level=logging.WARNING,
    )
    richardson_free = Richardson(
        codomain_free, stepsize=richardson_stepsize,
        op_norm=case["self_adjoint_op_norm"],
        change_init=False, logging_level=logging.WARNING,
    )
    x_richardson_standard, _ = run_steps(
        richardson_standard, EQUIVALENCE_STEPS
    )
    x_richardson_free, _ = run_steps(
        richardson_free, EQUIVALENCE_STEPS
    )

    spectral_richardson_standard = SpectralSelfAdjoint_Richardson(
        standard, k=EQUIVALENCE_STEPS,
        stepsize=richardson_stepsize,
    )
    spectral_richardson_free = SpectralSelfAdjoint_Richardson(
        codomain_free, k=EQUIVALENCE_STEPS,
        stepsize=richardson_stepsize,
    )
    x_spectral_richardson_standard, _ = run_steps(
        spectral_richardson_standard, 1
    )
    x_spectral_richardson_free, _ = run_steps(
        spectral_richardson_free, 1
    )
    record_strict_comparison(
        spectral_iteration_failures,
        '  Richardson standard: iterative vs spectral',
        case["h_domain"],
        x_richardson_standard, x_spectral_richardson_standard,
    )
    record_strict_comparison(
        spectral_iteration_failures,
        '  Richardson codomain-free: iterative vs spectral',
        case["h_domain"],
        x_richardson_free, x_spectral_richardson_free,
    )

if spectral_iteration_failures:
    ALL_FAILURES.extend(
        'spectral iteration: ' + failure
        for failure in spectral_iteration_failures
    )
    print('\nSpectral iteration failures were recorded; continuing.')
else:
    print('\nAll iterative/spectral comparisons passed.')


real, self-adjoint
  Landweber standard: iterative vs spectral                              2.407e-16
  Landweber codomain-free: iterative vs spectral                         2.539e-15
  Richardson standard: iterative vs spectral                             4.040e-16
  Richardson codomain-free: iterative vs spectral                        4.138e-16

real, non-self-adjoint
  Landweber standard: iterative vs spectral                              1.290e-15
  Landweber codomain-free: iterative vs spectral                         2.599e-15
  Richardson: SKIP (operator is not self-adjoint)

complex, self-adjoint
  Landweber standard: iterative vs spectral                              1.900e-16
  Landweber codomain-free: iterative vs spectral                         2.099e-15
  Richardson standard: iterative vs spectral                             4.760e-16
  Richardson codomain-free: iterative vs spectral                        4.480e-16

complex, non-self-adjoint
  Landweber standard: iter

## Tikhonov-CG stopping-rule guarantees

A spectral Tikhonov solution with $\alpha=10^{-4}$ is used as reference. The absolute-domain, relative-domain, and relative-codomain stopping rules are tested separately. The codomain-free solver has no relative-codomain stopping rule, so only its two domain criteria are tested. The internal CG iterations are allowed up to 1000 steps.

In [6]:
tikhonov_tolerance_failures = []

for case in CASES:
    print(f'\n{case["name"]}')
    spectral_standard = Spectral_Tikhonov(make_setting(case), regpar=REGPAR)
    spectral_free = Spectral_Tikhonov(
        make_setting(case, codomain_free=True), regpar=REGPAR
    )
    x_reference_standard, _ = run_steps(spectral_standard, 1)
    x_reference_free, _ = run_steps(spectral_free, 1)
    y_reference_standard = case["op"](x_reference_standard)

    variants = [
        (False, x_reference_standard, y_reference_standard),
        (True, x_reference_free, None),
    ]
    for is_free, x_reference, y_reference in variants:
        variant_name = 'codomain-free' if is_free else 'standard'
        rule_specs = [
            ('absolute domain', TIKHONOV_ABS_TOL, {
                'tol': TIKHONOV_ABS_TOL, 'reltolx': None, 'reltoly': None,
            }),
            ('relative domain', TIKHONOV_REL_TOL, {
                'tol': None, 'reltolx': TIKHONOV_REL_TOL, 'reltoly': None,
            }),
        ]
        if not is_free:
            rule_specs.append(('relative codomain', TIKHONOV_REL_TOL, {
                'tol': None, 'reltolx': None, 'reltoly': TIKHONOV_REL_TOL,
            }))

        for rule_name, requested_tolerance, solver_parameters in rule_specs:
            setting = make_setting(case, codomain_free=is_free, regpar=REGPAR)
            solver = TikhonovCG(
                setting, logging_level=logging.WARNING, x_exact=x_reference,
                y_exact=y_reference, **solver_parameters,
            )
            x_approx, y_approx = solver.run(max_it=TIKHONOV_MAX_IT)
            if rule_name == 'absolute domain':
                measured_error = difference_norm(
                    case["h_domain"], x_approx, x_reference
                )
            elif rule_name == 'relative domain':
                reference_increment = difference_norm(
                    case["h_domain"], x_reference, case["init"]
                )
                measured_error = (
                    difference_norm(case["h_domain"], x_approx, x_reference)
                    / max(reference_increment, np.finfo(float).tiny)
                )
            else:
                initial_image = case["op"](case["init"])
                reference_increment = difference_norm(
                    case["h_codomain"], y_reference, initial_image
                )
                measured_error = (
                    difference_norm(case["h_codomain"], y_approx, y_reference)
                    / max(reference_increment, np.finfo(float).tiny)
                )
            passed = measured_error <= requested_tolerance * (1 + 1e-12)
            print(
                f'  {variant_name:<15} {rule_name:<20} '
                f'iterations={solver.iteration_step_nr:4d}  '
                f'error={measured_error:.3e}  bound={requested_tolerance:.1e}  '
                f'{"PASS" if passed else "FAIL"}'
            )
            if not passed:
                tikhonov_tolerance_failures.append(
                    f'{case["name"]}, {variant_name}, {rule_name}: '
                    f'{measured_error:.3e} > {requested_tolerance:.1e}'
                )

if tikhonov_tolerance_failures:
    ALL_FAILURES.extend(
        'Tikhonov stopping rule: ' + failure
        for failure in tikhonov_tolerance_failures
    )
    print('\nTikhonov stopping-rule failures were recorded; continuing.')
else:
    print('\nAll requested Tikhonov-CG stopping-rule bounds passed.')


real, self-adjoint
  standard        absolute domain      iterations=  86  error=7.116e-09  bound=1.0e-04  PASS
  standard        relative domain      iterations=  79  error=6.611e-09  bound=1.0e-04  PASS
  standard        relative codomain    iterations=  32  error=4.128e-07  bound=1.0e-04  PASS
  codomain-free   absolute domain      iterations=  86  error=7.433e-09  bound=1.0e-04  PASS
  codomain-free   relative domain      iterations=  81  error=6.579e-09  bound=1.0e-04  PASS

real, non-self-adjoint
  standard        absolute domain      iterations=  86  error=7.454e-09  bound=1.0e-04  PASS
  standard        relative domain      iterations=  81  error=6.586e-09  bound=1.0e-04  PASS
  standard        relative codomain    iterations=  33  error=4.079e-07  bound=1.0e-04  PASS
  codomain-free   absolute domain      iterations=  86  error=7.454e-09  bound=1.0e-04  PASS
  codomain-free   relative domain      iterations=  81  error=6.583e-09  bound=1.0e-04  PASS

complex, self-adjoint
  s

## Alternatives for minimizing quadratic Tikhonov functionals

For the complex self-adjoint and the real non-self-adjoint operator, the methods `dual_TikhCG`, `FB`, `dual_FB`, `FISTA`, `dual_FISTA`, `PDHG`, and `dual_PDHG` are run for 50 iterations. Their results are compared with the exact minimizer of the discrete Tikhonov functional computed by the spectral method. Both the absolute and relative differences in the domain Hilbert-space norm are reported.

In [7]:
QUADRATIC_TIKHONOV_METHODS = (
    'dual_TikhCG',
    'FB',
    'dual_FB',
    'FISTA',
    'dual_FISTA',
    'PDHG',
    'dual_PDHG',
)
QUADRATIC_TIKHONOV_CASES = {
    'complex, self-adjoint',
    'real, non-self-adjoint',
}
quadratic_tikhonov_results = []

for case in CASES:
    if case['name'] not in QUADRATIC_TIKHONOV_CASES:
        continue

    reference_solver = Spectral_Tikhonov(
        make_setting(case), regpar=REGPAR,
    )
    x_reference, _ = run_steps(reference_solver, 1)
    print(f'\n{case["name"]}')
    print(f'{"method":<18} {"absolute difference":>21} {"relative difference":>21}')

    for method_name in QUADRATIC_TIKHONOV_METHODS:
        setting = make_setting(case, regpar=REGPAR)
        method_parameters = {}
        if method_name == 'dual_TikhCG':
            # Suppress TikhonovCG's tolerance criteria so that the
            # common CountIterations rule determines the run length.
            method_parameters = {
                'tol': 0.0, 'reltolx': None, 'reltoly': None,
            }
        x_approx, _ = setting.run(
            method_name,
            stoprule=CountIterations(OTHER_ITERATION_STEPS),
            **method_parameters,
        )
        absolute_error = difference_norm(
            case['h_domain'], x_approx, x_reference,
        )
        relative_error = relative_difference(
            case['h_domain'], x_approx, x_reference,
        )
        quadratic_tikhonov_results.append({
            'case': case['name'],
            'method': method_name,
            'iterations': OTHER_ITERATION_STEPS,
            'absolute_difference': absolute_error,
            'relative_difference': relative_error,
        })
        print(
            f'{method_name:<18} {absolute_error:21.3e} {relative_error:21.3e}'
        )


real, non-self-adjoint
method               absolute difference   relative difference


TypeError: '>=' not supported between instances of 'NoneType' and 'int'

## Approximation quality with method-specific iteration counts

The cell prints relative errors in the domain and codomain. Landweber uses 300 iterations; all other iterative methods use at most 50. Richardson and the self-adjoint $\nu$-method are selected through their applicability checks and thus run only for self-adjoint operators. CGNE is tested with exact data. All failures are collected so the complete table is printed before an exception is raised.

In [ ]:
quality_failures = []
quality_results = []

for case in CASES:
    setting = make_setting(case)
    mu_normal = case["landweber_stepsize"]
    mu_self_adjoint = (
        None if not case["self_adjoint"]
        else STEPSIZE_SAFETY / case["self_adjoint_op_norm"]
    )
    methods = [
        ('Landweber', Landweber, LANDWEBER_STEPS, lambda: Landweber(
            setting, stepsize=mu_normal, op_norm=case["op_norm"],
            change_init=False, logging_level=logging.WARNING,
        )),
        ('nu-method', NuMethod, OTHER_ITERATION_STEPS, lambda: NuMethod(
            setting, nu=NU, stepsize=mu_normal, op_norm=case["op_norm"],
            change_init=False, logging_level=logging.WARNING,
        )),
        ('CGNE', CGNE, OTHER_ITERATION_STEPS, lambda: CGNE(
            setting, change_init=False, logging_level=logging.WARNING,
        )),
        ('Richardson', Richardson, OTHER_ITERATION_STEPS, lambda: Richardson(
            setting, stepsize=mu_self_adjoint,
            op_norm=case.get("self_adjoint_op_norm"),
            change_init=False, logging_level=logging.WARNING,
        )),
        ('nu-method self-adjoint', NuMethodSelfAdjoint, OTHER_ITERATION_STEPS, lambda: NuMethodSelfAdjoint(
            setting, nu=NU, stepsize=mu_self_adjoint,
            op_norm=case.get("self_adjoint_op_norm"), change_init=False,
            logging_level=logging.WARNING,
        )),
    ]
    for method_name, solver_class, max_steps, construct_solver in methods:
        applicable = applicability(solver_class, setting)
        if not applicable["applicable"]:
            print(
                f'{case["name"]:<30} {method_name:<25} '
                f'SKIP ({applicable["info"].strip()})'
            )
            continue
        try:
            solver = construct_solver()
            approximation, performed = run_steps(solver, max_steps)
            image = case["op"](approximation)
            domain_error = relative_difference(
                case["h_domain"], approximation, case["x_exact"]
            )
            codomain_error = relative_difference(
                case["h_codomain"], image, case["data"]
            )
            quality_results.append({
                'operator': case["name"], 'method': method_name,
                'iterations': performed, 'domain_error': domain_error,
                'codomain_error': codomain_error,
            })
            print(
                f'{case["name"]:<30} {method_name:<25} '
                f'iterations={performed:2d}  domain={domain_error:.3e}  '
                f'codomain={codomain_error:.3e}'
            )
            if domain_error > DOMAIN_ERROR_LIMIT:
                quality_failures.append(
                    f'{case["name"]}, {method_name}: domain error '
                    f'{domain_error:.3e} > {DOMAIN_ERROR_LIMIT:.1e}'
                )
            if codomain_error > CODOMAIN_ERROR_LIMIT:
                quality_failures.append(
                    f'{case["name"]}, {method_name}: codomain error '
                    f'{codomain_error:.3e} > {CODOMAIN_ERROR_LIMIT:.1e}'
                )
        except Exception as error:
            quality_failures.append(
                f'{case["name"]}, {method_name}: '
                f'{type(error).__name__}: {error}'
            )
            print(
                f'{case["name"]:<30} {method_name:<25} '
                f'ERROR ({type(error).__name__}: {error})'
            )

if quality_failures:
    ALL_FAILURES.extend(
        'approximation quality: ' + failure for failure in quality_failures
    )
    print('\nApproximation-quality failures were recorded.')
else:
    print('\nAll applicable iterative methods satisfy the requested error bounds.')

In [ ]:
if ALL_FAILURES:
    print(f'\n{len(ALL_FAILURES)} test failure(s):')
    for number, failure in enumerate(ALL_FAILURES, start=1):
        print(f'{number:2d}. {failure}')
    raise AssertionError(
        f'{len(ALL_FAILURES)} linear Hilbert-solver test(s) failed.'
    )
print('\nAll linear Hilbert-solver tests passed.')